<a href="https://colab.research.google.com/github/hyunkyung31/DeepLearningProject/blob/main/hyunkyung/%EA%B3%BC%EC%A0%81%ED%95%A9_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/🐥new_2_project/04_DL_project/02_dataset/CADICA.zip'
extract_path = '/content/cadica_dataset/'

if os.path.exists(zip_path) :
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f'압축 해제 완료, 저장위치 : {extract_path}')
else :
    print(f'에러 {zip_path}에 파일이 존재하지 않습니다')

압축 해제 완료, 저장위치 : /content/cadica_dataset/


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# 실제 CADICA 위치에 맞게 하나만 골라서 수정
CANDIDATES = [
    Path("/content/cadica_dataset/CADICA/selectedVideos")
]

BASE = None
for p in CANDIDATES:
    if p.exists():
        BASE = p
        break

print("CADICA selectedVideos:", BASE)

# 팀 공통 split (있으면 무조건 재사용)
SPLIT_CANDIDATES = [
    Path("/content/drive/MyDrive/🐥new_2_project/04_DL_project/08_전처리/common_split.csv")
]
SPLIT_CSV = None
for p in SPLIT_CANDIDATES:
    if p.exists():
        SPLIT_CSV = p
        break

print("common_split.csv:", SPLIT_CSV)
assert BASE is not None, "CADICA selectedVideos 경로를 못 찾음 — CANDIDATES에 실제 경로 추가"
assert SPLIT_CSV is not None, "common_split.csv 못 찾음 — Drive에서 경로 알려주거나 복사해와"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CADICA selectedVideos: /content/cadica_dataset/CADICA/selectedVideos
common_split.csv: /content/drive/MyDrive/🐥new_2_project/04_DL_project/08_전처리/common_split.csv


In [ ]:
# @title Step 1. bbox 있는 프레임 목록 만들기 (det_df)
import pandas as pd
from PIL import Image

df = pd.read_csv(SPLIT_CSV)
print(df.head())
print(df["split"].value_counts())

rows = []
missing_img = 0

for r in df.itertuples():
    vid_dir = BASE / r.patient_id / r.video_id
    gt_dir = vid_dir / "groundtruth"
    input_dir = vid_dir / "input"
    if not gt_dir.exists():
        continue

    for gt_file in sorted(gt_dir.glob("*.txt")):
        # 예: p1_v2_00012.txt
        stem = gt_file.stem  # p1_v2_00012
        # 이미지 후보
        img_path = None
        for ext in [".png", ".jpg", ".jpeg", ".bmp"]:
            cand = input_dir / f"{stem}{ext}"
            if cand.exists():
                img_path = cand
                break
            # 어떤 데이터는 frame id만 있을 수 있음
            fid = stem.split("_")[-1]
            cand2 = input_dir / f"{fid}{ext}"
            if cand2.exists():
                img_path = cand2
                break

        if img_path is None:
            missing_img += 1
            continue

        rows.append({
            "patient_id": r.patient_id,
            "video_id": r.video_id,
            "frame_id": stem,
            "img_path": str(img_path),
            "bbox_path": str(gt_file),
            "split": r.split,
            "view": getattr(r, "view", "NA"),
        })

det_df = pd.DataFrame(rows)
print("det frames:", len(det_df), "| missing img:", missing_img)
print(det_df["split"].value_counts())
det_df.head()

  patient_id video_id       label view  n_frames  has_groundtruth  split
0         p1       v2      lesion  LCA        56             True  train
1         p1       v3      lesion  LCA        44             True  train
2         p1      v10      lesion  LCA        56             True  train
3         p1      v11      lesion  LCA        44             True  train
4         p1       v1  non-lesion  LCA        55            False  train
split
train    271
val       36
test      27
Name: count, dtype: int64
det frames: 3685 | missing img: 0
split
train    3077
val       335
test      273
Name: count, dtype: int64


,patient_id,video_id,frame_id,img_path,bbox_path,split,view
0,p1,v2,p1_v2_00015,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,LCA
1,p1,v2,p1_v2_00016,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,LCA
2,p1,v2,p1_v2_00017,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,LCA
3,p1,v2,p1_v2_00018,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,LCA
4,p1,v2,p1_v2_00019,/content/cadica_dataset/CADICA/selectedVideos/...,/content/cadica_dataset/CADICA/selectedVideos/...,train,LCA


In [ ]:
import os
import shutil

YOLO_ROOT = Path("/content/cadica_yolo")

# 깨끗이 다시 만들기
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for split in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

def parse_bbox_file(path: Path):
    """CADICA: 각 줄 x y w h category (pixel, top-left)"""
    boxes = []
    text = path.read_text().strip()
    if not text:
        return boxes
    for line in text.splitlines():
        parts = line.strip().replace(",", " ").split()
        if len(parts) < 4:
            continue
        x, y, w, h = map(float, parts[:4])
        boxes.append((x, y, w, h))
    return boxes

def cadica_to_yolo(x, y, w, h, img_w, img_h):
    cx = (x + w / 2.0) / img_w
    cy = (y + h / 2.0) / img_h
    return cx, cy, w / img_w, h / img_h

n_empty = 0
n_boxes = 0

for r in det_df.itertuples():
    img_path = Path(r.img_path)
    bbox_path = Path(r.bbox_path)
    split = r.split

    with Image.open(img_path) as im:
        img_w, img_h = im.size

    boxes = parse_bbox_file(bbox_path)
    if len(boxes) == 0:
        n_empty += 1

    yolo_lines = []
    for x, y, w, h in boxes:
        cx, cy, nw, nh = cadica_to_yolo(x, y, w, h, img_w, img_h)
        # clip
        cx = min(max(cx, 0.0), 1.0)
        cy = min(max(cy, 0.0), 1.0)
        nw = min(max(nw, 0.0), 1.0)
        nh = min(max(nh, 0.0), 1.0)
        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        n_boxes += 1

    out_img = YOLO_ROOT / "images" / split / f"{r.frame_id}.png"
    out_lbl = YOLO_ROOT / "labels" / split / f"{r.frame_id}.txt"

    if not out_img.exists():
        os.symlink(img_path.resolve(), out_img)

    out_lbl.write_text("\n".join(yolo_lines))

print("empty label files:", n_empty)
print("total boxes:", n_boxes)
for split in ["train", "val", "test"]:
    ni = len(list((YOLO_ROOT / "images" / split).glob("*")))
    nl = len(list((YOLO_ROOT / "labels" / split).glob("*.txt")))
    print(f"{split}: images={ni}, labels={nl}")

empty label files: 0
total boxes: 5835
train: images=3077, labels=3077
val: images=335, labels=335
test: images=273, labels=273


In [ ]:
yaml_path = YOLO_ROOT / "data.yaml"
yaml_path.write_text(f"""\
path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['lesion']
""")
print(yaml_path.read_text())

# 라벨 샘플 확인
sample = next((YOLO_ROOT / "labels" / "train").glob("*.txt"))
print("sample label:", sample.name)
print(sample.read_text()[:200])

path: /content/cadica_yolo
train: images/train
val: images/val
test: images/test
nc: 1
names: ['lesion']

sample label: p37_v1_00016.txt
0 0.529297 0.359375 0.156250 0.074219


In [ ]:
# @title pos-only 데이터셋

import shutil
from pathlib import Path

YOLO_ROOT = Path("/content/cadica_yolo")
POS_ROOT = Path("/content/cadica_yolo_posonly")

if POS_ROOT.exists():
    shutil.rmtree(POS_ROOT)

def has_box(lbl: Path) -> bool:
    return lbl.exists() and len(lbl.read_text().strip()) > 0

for split in ["train", "val", "test"]:
    src_img = YOLO_ROOT / "images" / split
    src_lbl = YOLO_ROOT / "labels" / split
    dst_img = POS_ROOT / "images" / split
    dst_lbl = POS_ROOT / "labels" / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    n_all, n_pos = 0, 0
    for img in src_img.glob("*"):
        n_all += 1
        lbl = src_lbl / f"{img.stem}.txt"
        if has_box(lbl):
            n_pos += 1
            # symlink로 충분
            (dst_img / img.name).symlink_to(img.resolve())
            (dst_lbl / lbl.name).symlink_to(lbl.resolve())
    print(f"{split}: all={n_all}, pos={n_pos}")

POS_YAML = POS_ROOT / "data.yaml"
POS_YAML.write_text(f"""\
path: {POS_ROOT}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['lesion']
""")
print(POS_YAML.read_text())

train: all=3077, pos=3077
val: all=335, pos=335
test: all=273, pos=273
path: /content/cadica_yolo_posonly
train: images/train
val: images/val
test: images/test
nc: 1
names: ['lesion']



In [ ]:
from pathlib import Path

ROOT = Path("/content/cadica_yolo")
TRAIN_IMG = ROOT / "train" / "images"
TRAIN_LBL = ROOT / "train" / "labels"
VAL_IMG = ROOT / "val" / "images"
VAL_LBL = ROOT / "val" / "labels"
TEST_IMG = ROOT / "test" / "images"
TEST_LBL = ROOT / "test" / "labels"

print(len(list(TRAIN_IMG.glob("*"))), len(list(TRAIN_LBL.glob("*.txt"))))

0 0


In [ ]:
# @title Ultralytics 설치
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.3 MB/s eta 0:00:00


In [ ]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 52.6/235.7 GB disk)


In [ ]:
# @title 과적합 학습시키기

from ultralytics import RTDETR
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/DL")

RUN_NAME = "train_overfit_posonly_v1"
POS_YAML = "/content/cadica_yolo_posonly/data.yaml"
FULL_YAML = "/content/cadica_yolo/data.yaml"

OVERFIT_KWARGS = dict(
    data=POS_YAML,
    name=RUN_NAME,
    project=str(BASE),
    epochs=100,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    amp=True,
    lr0=0.0005,
    lrf=0.01,
    cos_lr=False,
    warmup_epochs=3,
    weight_decay=0.0,
    dropout=0.0,
    patience=0,      # early stop 끔
    mosaic=0.0,
    close_mosaic=0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    scale=0.0,
    translate=0.0,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
)

model = RTDETR("rtdetr-l.pt")
model.train(**OVERFIT_KWARGS)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cadica_yolo_posonly/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=train_overfit_posonly_v1, nbs=64, nms=False, opset=None, optimize=False, opt

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      6.57G      1.143      1.049     0.4125          6        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.7it/s 12.5s
                   all        335        506      0.135     0.0909     0.0462     0.0114

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      6.96G     0.6881      1.104     0.2206         15        640: 0% ──────────── 0/385  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      6.96G      0.548      1.214     0.1413          5        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.7it/s 7.7s
                   all        335        506     0.0283      0.132     0.0324    0.00727

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100      7.01G     0.6759      1.155     0.2264         10        640: 0% ──────────── 0/385  0.7s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     44/100      7.01G     0.1054     0.1965    0.02454         10        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:10
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.7it/s 7.7s
                   all        335        506      0.139      0.132     0.0526     0.0153

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     45/100      7.01G    0.08817     0.1704    0.01909         16        640: 0% ──────────── 0/385  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     45/100      7.01G    0.09347     0.1799    0.02201         11        640: 44% ━━━━━─────── 173/385 4.1it/s 1:54<52.1s


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
BASE = Path("/content/drive/MyDrive/DL/train_overfit_posonly_v1")
print((BASE / "weights" / "last.pt").exists())
print((BASE / "weights" / "best.pt").exists())

True
True


In [ ]:
from ultralytics import RTDETR

model = RTDETR("/content/drive/MyDrive/DL/train_overfit_posonly_v1/weights/last.pt")
r = model.val(data="/content/cadica_yolo_posonly/data.yaml", split="train", imgsz=640)
print(f"train P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f} mAP50-95={r.box.map:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2487.0±894.8 MB/s, size: 100.4 KB)
val: Scanning /content/cadica_yolo_posonly/labels/train.cache... 3077 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3077/3077 921.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 193/193 1.3it/s 2:33
                   all       3077       5056      0.993      0.996      0.992      0.947
Speed: 1.0ms preprocess, 45.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val
train P=0.993 R=0.996 mAP50=0.992 mAP50-95=0.947


In [ ]:
for split in ["val", "test"]:
    r = model.val(data="/content/cadica_yolo_posonly/data.yaml", split=split, imgsz=640)
    print(f"{split:5s} P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f}")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1680.8±285.9 MB/s, size: 91.8 KB)
val: Scanning /content/cadica_yolo_posonly/labels/val.cache... 335 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 335/335 87.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 1.2it/s 17.6s
                   all        335        506      0.139      0.138     0.0528     0.0154
Speed: 1.3ms preprocess, 45.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /content/runs/detect/val-2
val   P=0.139 R=0.138 mAP50=0.053
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 49

Mild 학습 1회 (지금 바로)
규제 과하게 말고, baseline에서만 살짝:

cos_lr=True, lr0=0.0003 정도
mosaic/dropout은 baseline 유지
→ “수렴만 조금 부드럽게”

In [ ]:
from ultralytics import RTDETR
from pathlib import Path

BASE = Path("/content/drive/MyDrive/DL")
DATA = "/content/cadica_yolo/data.yaml"
RUN = "train_mild_v1"

model = RTDETR("rtdetr-l.pt")
model.train(
    data=DATA,
    name=RUN,
    project=str(BASE),
    epochs=60,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    amp=True,
    lr0=0.0003,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,
    dropout=0.0,
    patience=25,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    scale=0.5,
    translate=0.1,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/cadica_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_mild_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, over

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/60      8.17G      1.321     0.7565     0.4141          6        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.6it/s 8.0s
                   all        335        506    0.00637      0.502    0.00497    0.00108

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/60      6.76G      1.036     0.7798     0.2705         25        640: 0% ──────────── 0/385  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/60      6.76G     0.8133       1.02     0.2023          7        640: 100% ━━━━━━━━━━━━ 385/385 1.4it/s 4:28
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.7it/s 7.7s
                   all        335        506     0.0152      0.138     0.0105    0.00367

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/60      6.81G      0.447     0.4443     0.1022         11        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.7it/s 7.8s
                   all        335        506      0.151      0.166     0.0559     0.0136

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      31/60      6.81G     0.5679     0.4284     0.1293         15        640: 0% ──────────── 0/385  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      31/60      6.81G     0.4545     0.4393     0.1062         10        640: 100% ━━━━━━━━━━━━ 385/385 1.5it/s 4:25
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.7it/s 7.7s
                   all        335        506      0.164      0.186      0.066     0.0169
EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 6, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

31 epochs completed in 2.384 hours.
Optimizer stripped from /content/drive/MyDrive/DL/train_mild_v1/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/DL/train_mild_v1/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/DL/train_mild_v1/weights/best.pt...
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a079dfc9790>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
from ultralytics import RTDETR
from pathlib import Path

BASE = Path("/content/drive/MyDrive/DL")
DATA = "/content/cadica_yolo/data.yaml"
weights = BASE / "train_mild_v1" / "weights" / "best.pt"

model = RTDETR(str(weights))
test = model.val(data=DATA, split="test", imgsz=640)

print(f"mild-test  P={test.box.mp:.3f} R={test.box.mr:.3f} "
      f"mAP50={test.box.map50:.3f} mAP50-95={test.box.map:.3f}")
print("baseline-test: P=0.592 R=0.289 mAP50=0.308 mAP50-95=0.150")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1547.2±657.8 MB/s, size: 95.7 KB)
val: Scanning /content/cadica_yolo/labels/test... 273 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 273/273 1.4Kit/s 0.2s
val: New cache created: /content/cadica_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.5it/s 11.7s
                   all        273        273      0.581      0.304      0.291      0.113
Speed: 3.0ms preprocess, 35.7ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val-4
mild-test  P=0.581 R=0.304 mAP50=0.291 mAP50-95=0.113
baseline-test: P=0.592 R=0.289 mAP50=0.308 mAP50-95=0.150


In [ ]:
# @title conf 스윕 (baseline에 대해)
from ultralytics import RTDETR
from pathlib import Path
import pandas as pd

DATA = "/content/cadica_yolo/data.yaml"
WEIGHTS = "/content/drive/MyDrive/🐥new_2_project/04_DL_project/RT-DETR_result/train_v3_amp_on/weights/best.pt"
model = RTDETR(WEIGHTS)

ref = model.val(data=DATA, split="test", imgsz=640, plots=False, verbose=False)
print(f"[REF] mAP50={ref.box.map50:.3f} P={ref.box.mp:.3f} R={ref.box.mr:.3f}")

rows = [{"conf": "default", "iou": "default", "P": float(ref.box.mp), "R": float(ref.box.mr),
         "mAP50": float(ref.box.map50), "mAP50-95": float(ref.box.map)}]
for conf in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
    for iou in [0.5, 0.6, 0.7]:
        r = model.val(data=DATA, split="test", imgsz=640, conf=conf, iou=iou, plots=False, verbose=False)
        rows.append({"conf": conf, "iou": iou, "P": float(r.box.mp), "R": float(r.box.mr),
                     "mAP50": float(r.box.map50), "mAP50-95": float(r.box.map)})

df = pd.DataFrame(rows)
print(df.sort_values("mAP50", ascending=False))

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1505.1±370.5 MB/s, size: 102.0 KB)
val: Scanning /content/cadica_yolo/labels/test.cache... 273 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 273/273 63.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.7it/s 10.9s
                   all        273        273      0.592      0.289      0.308       0.15
Speed: 1.1ms preprocess, 35.1ms inference, 0.0ms loss, 0.2ms postprocess per image
[REF] mAP50=0.308 P=0.592 R=0.289
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1553.5±880.9 MB/s, size: 113.0 KB)
val: Scanning /content/cadica_yolo/